# MultiModal RAG — Interactive Demo

This notebook demonstrates the full MultiModal RAG pipeline with local LM Studio integration:
1. Document ingestion (PDF/PPTX → images)
2. LLM-based XML transcription (images → structured XML)
3. Semantic chunking (XML → semantic chunks)
4. Vector indexing (chunks → vector database)
5. RAG querying (query → answer with sources)

## Setup

First, ensure you have the required dependencies installed:

```bash
pip install -e ".[dev]"
```

### Configure LM Studio

1. Start LM Studio and load a model in the **Local Server** tab
2. Note the model name (e.g., `Qwen/Qwen2.5-72B-Instruct`)
3. Copy `.env.example` to `.env` and update:

```env
LM_STUDIO_API_BASE=http://localhost:1234/v1
LM_STUDIO_MODEL=Qwen/Qwen2.5-72B-Instruct  # Your model
```

The pipeline automatically picks up LM Studio settings from `.env`.

In [1]:
import os
import sys
from pathlib import Path

# Add project root
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv()

from multimodal_rag import MultiModalRAGPipeline
from multimodal_rag.utils.config import Config

## Step 1: Detect Your API Configuration

The pipeline automatically detects which API is configured. Let's check:

In [2]:
# Detect which API is configured
lm_studio_base = os.getenv('LM_STUDIO_API_BASE')
freellm_base = os.getenv('FREELLM_API_BASE')
openai_key = os.getenv('OPENAI_API_KEY')

if lm_studio_base:
    print(f'🎓 Using LM Studio: {lm_studio_base}')
    model = os.getenv('LM_STUDIO_MODEL', 'not set')
    print(f'   Model: {model}')
elif freellm_base:
    print(f'📡 Using FreeLLMAPI: {freellm_base}')
elif openai_key:
    print('☁️  Using OpenAI Cloud API')
else:
    print('⚠️  No API configured.')
    print('   Set LM_STUDIO_API_BASE in your .env file')

print('\n✅ Configuration detected successfully')

🎓 Using LM Studio: http://localhost:1234/v1
   Model: qwen/qwen3.6-35b-a3b

✅ Configuration detected successfully


## Step 2: Initialize the Pipeline

We can use default configuration or customize any parameter. The pipeline will automatically use your LM Studio settings.

In [6]:
# Initialize pipeline with default config (uses LM Studio from .env)
pipeline = MultiModalRAGPipeline()
print('✅ Pipeline initialized')

# View current configuration
config = pipeline.config
print(f'\n📋 Current Configuration:')
print(f'   LLM Model: {config.llm.model}')
print(f'   LLM Base URL: {config.llm.base_url}')
print(f'   Embedding: {config.embedding.model}')
print(f'   Vector Store: {config.indexing.vector_store}')
print(f'   Max Concurrency: {config.transcription.max_concurrent}')

# Show which API is being used
if config.llm.base_url:
    print(f'\n📡 API Endpoint: {config.llm.base_url}')
else:
    print('\n⚠️  No API endpoint configured')

2026-10-25 14:38:54 | multimodal_rag.ingestion.ingestion | INFO | DocumentIngestor initialized: DPI=300, format=png, max_pages=None
2026-10-25 14:38:54 | multimodal_rag.transcription.transcription | INFO | XMLTranscriber initialized: model=qwen/qwen3.6-35b-a3b, concurrency=10
2026-10-25 14:38:54 | multimodal_rag.chunking.chunking | INFO | SemanticChunker initialized: chunk_size=512, overlap=50, min_chunk=100
2026-10-25 14:38:54 | multimodal_rag.indexing.indexing | INFO | VectorIndexer initialized: backend=chroma, collection=multimodal_rag_collection
2026-10-25 14:38:54 | multimodal_rag.rag.rag | INFO | RAGEngine initialized: model=qwen/qwen3.6-35b-a3b, top_k=5
2026-10-25 14:38:54 | multimodal_rag.pipeline.pipeline | INFO | MultiModalRAGPipeline initialized
2026-10-25 14:38:54 | multimodal_rag.pipeline.pipeline | INFO |   LLM Model: qwen/qwen3.6-35b-a3b
  Embedding: BAAI/bge-large-en-v1.5  Vector Store: chroma
  Max Concurrency: 10
✅ Pipeline initialized

📋 Current Configuration:
   LLM

## Step 3: Test LM Studio Connection

Let's verify your LM Studio server is running and responding:

In [7]:
from openai import OpenAI

lm_studio_base = os.getenv('LM_STUDIO_API_BASE')
lm_studio_model = os.getenv('LM_STUDIO_MODEL', 'Qwen/Qwen2.5-72B-Instruct')

if not lm_studio_base:
    print('⚠️  LM Studio not configured. Set LM_STUDIO_API_BASE in .env')
else:
    try:
        client = OpenAI(
            base_url=lm_studio_base,
            api_key="lm-studio"  # LM Studio uses this dummy key
        )
        
        # Test the connection
        response = client.chat.completions.create(
            model=lm_studio_model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "Say hello in one sentence."}
            ],
            temperature=0.7,
        )
        
        print(f'✅ LM Studio is responding!')
        print(f'   Model: {response.model}')
        print(f'   Response: {response.choices[0].message.content}')
        
    except Exception as e:
        print(f'❌ Connection failed: {e}')
        print('   Make sure LM Studio server is running on the configured port')

✅ LM Studio is responding!
   Model: qwen/qwen3.6-35b-a3b
   Response: 

Hello, and I'm here to help you with anything you need.


## Step 4: Process a Document

Place your PDF or PPTX file in `examples/sample_documents/` and run this cell.

In [13]:
from pathlib import Path

sample_dir = Path('/Users/apple/Downloads')
documents = list(sample_dir.glob('*.pdf')) + list(sample_dir.glob('*.pptx'))

if documents:
    doc_path = str(documents[0])
    print(f'Processing: {doc_path}')
    
    result = pipeline.process_document(doc_path)
    print(f'\n✅ Processing Result:')
    print(f'   Status: {result["status"]}')
    print(f'   Pages: {result.get("pages", 0)}')
    print(f'   Chunks: {result.get("total_chunks", 0)}')
else:
    print(f'No documents found in {sample_dir}/')
    print('Place a PDF or PPTX file there and try again.')

Processing: /Users/apple/Downloads/tramchandani-automating-web-190625.pdf
2026-10-25 14:39:33 | multimodal_rag.pipeline.pipeline | INFO | Processing document: /Users/apple/Downloads/tramchandani-automating-web-190625.pdf
2026-10-25 14:39:33 | multimodal_rag.pipeline.pipeline | INFO | Step 1/4: Ingesting pages...
2026-10-25 14:39:33 | multimodal_rag.ingestion.ingestion | INFO | Ingesting document: /Users/apple/Downloads/tramchandani-automating-web-190625.pdf
2026-10-25 14:39:59 | multimodal_rag.ingestion.ingestion | INFO | Converted 12 PDF pages to images
2026-10-25 14:39:59 | multimodal_rag.ingestion.ingestion | INFO | Successfully ingested 12 pages from /Users/apple/Downloads/tramchandani-automating-web-190625.pdf
2026-10-25 14:39:59 | multimodal_rag.pipeline.pipeline | INFO | Step 2/4: Transcribing pages to XML...
2026-10-25 14:39:59 | multimodal_rag.transcription.transcription | INFO | Starting transcription of 12 pages (concurrency=10)
2026-10-25 14:39:59 | multimodal_rag.transcrip

In [14]:
result = pipeline.query("What is the main topic of this document?")

print("Result type:", type(result))
print("Answer:", repr(result.answer))
print("Metadata:", result.metadata)
print("Sources:", len(result.sources) if result.sources else 0)

# Inspect every available field on the result object
if hasattr(result, "model_dump"):
    print("Full result:", result.model_dump())
elif hasattr(result, "__dict__"):
    print("Full result:", vars(result))

2026-10-25 14:44:18 | multimodal_rag.rag.rag | INFO | Processing query: 'What is the main topic of this document?'
2026-10-25 14:44:41 | multimodal_rag.rag.rag | INFO | LLM response: model=qwen/qwen3.6-35b-a3b, finish_reason=stop, content='\n\nBased on the provided context, the main topic of this document is **Agentic AI** (or AI Agents). The excerpts highlight key aspects of this field, including:\n- Search interest trends comparing "AI Agents" and "Agentic AI" (Page 3)\n- Reasoning frameworks like Graph of Thought (GoT) for parallel subgoal planning (Page 8)\n- General relationships within the field (Page 5)\n\nAll provided excerpts are sourced from `tramchandani-automating-web-190625.pdf`.'
2026-10-25 14:44:41 | multimodal_rag.rag.rag | INFO | Query answered with 5 sources
Result type: <class 'multimodal_rag.rag.rag.RAGResult'>
Answer: 'Based on the provided context, the main topic of this document is **Agentic AI** (or AI Agents). The excerpts highlight key aspects of this field, i

## Step 5: Query the Processed Documents

Try different queries to see how the RAG engine retrieves and answers.

In [17]:
queries = [
    'What is the main topic of this document?',
    'Summarize the key points.',
    'What methodology or approach was used?',
]

for query in queries:
    print(f'\n❓ Query: "{query}"')
    print('-' * 50)
    
    result = pipeline.query(str(query))
    print(f'📝 Answer: {result.answer}')
    print(f'📊 Sources used: {result.metadata.get("num_sources", 0)}')
    
    if result.sources:
        print('\n📚 Top Sources:')
        for i, source in enumerate(result.sources[:3], 1):
            page = source.get('page_number', 'N/A')
            content = source.get('content', '')[:200]
            print(f'   [{i}] Page {page}: {content}...')


❓ Query: "What is the main topic of this document?"
--------------------------------------------------
2026-10-25 14:44:46 | multimodal_rag.rag.rag | INFO | Processing query: 'What is the main topic of this document?'
2026-10-25 14:45:06 | multimodal_rag.rag.rag | INFO | LLM response: model=qwen/qwen3.6-35b-a3b, finish_reason=stop, content='\n\nBased on the provided context, the main topic of the document is **Agentic AI** (or AI Agents). This is directly supported by:\n- The explicit mention of "Agentic AI" in the footer of page 5.\n- A Google Trends chart on page 3 tracking and comparing the rising search interest for "AI Agents" and "Agentic AI" from late 2022 through 2025.\n- Technical content on page 8 discussing AI planning and reasoning frameworks, specifically "Graph of Thought (GoT)" and "parallel subgoal reasoning."\n\nTogether, these excerpts indicate the document focuses on the trends, conceptual frameworks, and reasoning mechanisms of Agentic AI.'
2026-10-25 14:45:06 | mu

## Step 6: Inspect Pipeline State

View all intermediate results and pipeline statistics.

In [18]:
# Pipeline status
status = pipeline.get_status()
print('Pipeline Status:')
print(f'  Documents processed: {status.documents_processed}')
print(f'  Total pages: {status.total_pages}')
print(f'  Total chunks: {status.total_chunks}')
print(f'  Indexed: {status.is_indexed}')

# Index info
index_info = pipeline.get_index_info()
print(f'\nIndex Info: {index_info}')

# Access transcribed documents (XML)
docs = pipeline.get_transcribed_documents()
print(f'\nTranscribed Documents: {len(docs)}')
for doc in docs:
    print(f'  - {doc.source_file}: {len(doc.pages)} pages')

# Access chunks
chunks = pipeline.get_chunks()
print(f'\nSemantic Chunks: {len(chunks)}')
for chunk in chunks[:3]:  # Show first 3
    print(f'  [{chunk.chunk_id}] ({chunk.token_count} tokens)')
    print(f'    {chunk.content[:150]}...')

Pipeline Status:
  Documents processed: 1
  Total pages: 12
  Total chunks: 22
  Indexed: True

Index Info: {'collection_name': 'multimodal_rag_collection', 'total_vectors': 22, 'backend': 'chroma'}

Transcribed Documents: 1
  - /Users/apple/Downloads/tramchandani-automating-web-190625.pdf: 12 pages

Semantic Chunks: 22
  [/Users/apple/Downloads/tramchandani-automating-web-190625.pdf_p1_chunk_0001] (25 tokens)
    <document>
    <heading level="1">Agentic AI</heading>
    <text>Toni Ramchandani</text>
</document>...
  [/Users/apple/Downloads/tramchandani-automating-web-190625.pdf_p2_chunk_0001] (47 tokens)
    <document>
    <figure>
        <description>Portrait of a man with dark hair and a beard, wearing sunglasses, a white shirt, a dark tie, and a dark s...
  [/Users/apple/Downloads/tramchandani-automating-web-190625.pdf_p2_chunk_0002] (83 tokens)
    <h2>A Passionate Technocrat</h2>
        <ul>
            <li>AI Research</li>
            <li>Automation</li>
            <li>Autho

In [19]:
import multimodal_rag.pipeline.pipeline as module
import inspect

print("Loaded module:", module.__file__)

print("get_status line:",
      module.MultiModalRAGPipeline.get_status.__code__.co_firstlineno)

print("get_status bytecode names:",
      module.MultiModalRAGPipeline.get_status.__code__.co_names)

print("get_status result:",
      pipeline.get_status())

Loaded module: /Users/apple/Documents/Projects/MultiRAG/multimodal_rag/pipeline/pipeline.py
get_status line: 389
get_status bytecode names: ('replace', '_status')
get_status result: PipelineStatus(documents_processed=1, total_pages=12, total_chunks=22, is_indexed=True, last_error=None)


## Step 7: Export Results

Export transcribed XML and semantic chunks for further analysis.

In [ ]:
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

# Export XML files
pipeline.export_xml(str(output_dir / 'xml'))
print(f'✅ XML exported to {output_dir / "xml"}')

# Export chunks as JSON
pipeline.export_chunks(str(output_dir / 'chunks'))
print(f'✅ Chunks exported to {output_dir / "chunks"}')

# Save vector index
pipeline.save_index(str(output_dir / 'index'))
print(f'✅ Index saved to {output_dir / "index"}')

## Custom Configuration

You can customize any aspect of the pipeline. Here's an example with custom settings for local models.

In [ ]:
from multimodal_rag.utils.config import (
    Config, LLMConfig, EmbeddingConfig, IngestionConfig,
    TranscriptionConfig, ChunkingConfig, IndexingConfig, RAGConfig
)

# Create custom configuration for local models
custom_config = Config()

# Set models for LM Studio (update with your loaded models)
custom_config.llm = LLMConfig(
    model=os.getenv('LM_STUDIO_MODEL', 'Qwen/Qwen2.5-72B-Instruct'),
    base_url=os.getenv('LM_STUDIO_API_BASE', 'http://localhost:1234/v1'),
    api_key=os.getenv('LM_STUDIO_API_KEY', 'lm-studio'),
)

custom_config.embedding = EmbeddingConfig(
    model=os.getenv('LM_STUDIO_EMBEDDING_MODEL', 'bge-large-en-v1.5'),
    base_url=os.getenv('LM_STUDIO_API_BASE', 'http://localhost:1234/v1'),
)

custom_config.ingestion = IngestionConfig(dpi=300, max_pages=10)
custom_config.transcription = TranscriptionConfig(
    max_concurrent=5,
    model=os.getenv('LM_STUDIO_VISION_MODEL', 'Qwen/Qwen2-VL-72B-Instruct'),
)
custom_config.indexing = IndexingConfig(
    vector_store='chroma',
    collection_name='my_collection',
    top_k=5
)
custom_config.rag = RAGConfig(
    llm_model=os.getenv('LM_STUDIO_MODEL', 'Qwen/Qwen2.5-72B-Instruct'),
    top_k=5
)

# Initialize pipeline with custom config
custom_pipeline = MultiModalRAGPipeline(custom_config)
print('✅ Custom pipeline initialized')
print(f'   Using models from your LM Studio configuration')